In [1]:
import numpy as np
import pandas as pd
import sys
sys.path.insert(0, '.')

from Engine.TickData import TickData
from Engine.Backtester import Backtester

In [2]:
# Load ES tick data and aggregate into 250-tick bars
td = TickData('./Data/ESTick.txt', timezone='America/Chicago')
td.aggregate(method='tick', size=250)

tick_value = 12.50
contract_multiplier = 50
initial_equity = 100_000

ohlc = td.get_ohlc()
indices = td.get_indices()
timeframe = td.get_timeframe()
n_bars = len(ohlc)
close = ohlc[:, 3]

print(f'Bars: {n_bars:,}')
print(f'Date range: {indices[0]} to {indices[-1]}')


Loading tick data: ./Data/ESTick.txt
Loaded 23,033,489 ticks
Price range: 6751.50 - 7027.25

Aggregated 23,033,489 ticks into 92,133 bars (250 tick)
Bars: 92,133
Date range: 2026-02-01 23:00:00.003520-06:00 to 2026-03-02 07:30:20.001880-06:00


In [3]:
def ultimate_smoother(price: np.ndarray, period: float) -> np.ndarray:
    """Ehlers Ultimate Smoother - zero lag in pass band."""
    n = len(price)
    us = np.copy(price).astype(float)

    a1 = np.exp(-1.414 * np.pi / period)
    b1 = 2 * a1 * np.cos(1.414 * np.pi / period)
    c2 = b1
    c3 = -a1 * a1
    c1 = (1 + c2 - c3) / 4

    for i in range(3, n):
        us[i] = ((1 - c1) * price[i]
                 + (2 * c1 - c2) * price[i - 1]
                 - (c1 + c3) * price[i - 2]
                 + c2 * us[i - 1]
                 + c3 * us[i - 2])

    return us

In [ ]:
us = ultimate_smoother(close, period=20.0)

# =====================================================================
# 2. VECTOR ALIGNMENT (PREVENTING LOOKAHEAD BIAS)
# =====================================================================
# Time T-1: us[:-2] (Used to calculate slope at Time T)
# Time T:   us[1:-1], t_current_close, t_current_open (Current State)
# Time T+1: t_next_close, t_next_open (Forward Outcome)

t_current_close = close[1:-1]
t_next_close = close[2:]
t_current_open = ohlc[1:-1, 0]
t_next_open = ohlc[2:, 0]

# =====================================================================
# TEST 1: ULTIMATE SMOOTHER SLOPE PREDICTION
# =====================================================================
t_us_slope = us[1:-1] - us[:-2]
t_fwd_move_ticks = (t_next_close - t_current_close) / 0.25

us_up = t_us_slope > 0
us_down = t_us_slope < 0

prob_up_cont = np.mean(t_fwd_move_ticks[us_up] > 0)
ev_up_cont = np.mean(t_fwd_move_ticks[us_up])

prob_down_cont = np.mean(t_fwd_move_ticks[us_down] < 0)
ev_down_cont = np.mean(-t_fwd_move_ticks[us_down]) 

print("="*55)
print("TEST 1: ULTIMATE SMOOTHER SLOPE (1-Bar Forward)")
print("="*55)
print(f"When US Slope is UP   -> Next Close is Higher: {prob_up_cont*100:.2f}% | EV: {ev_up_cont:.4f} ticks")
print(f"When US Slope is DOWN -> Next Close is Lower:  {prob_down_cont*100:.2f}% | EV: {ev_down_cont:.4f} ticks")


# =====================================================================
# TEST 2: 3-STATE CANDLE COLOR MARKOV CHAIN
# =====================================================================
# Define the 3 States at Time T
is_green = t_current_close > t_current_open
is_red = t_current_close < t_current_open
is_doji = t_current_close == t_current_open

# Define the 3 Outcomes at Time T+1
next_green = t_next_close > t_next_open
next_red = t_next_close < t_next_open
next_doji = t_next_close == t_next_open

# Calculate Transition Matrix Probabilities
# --- From GREEN ---
p_g_g = np.mean(next_green[is_green])
p_g_r = np.mean(next_red[is_green])
p_g_d = np.mean(next_doji[is_green])

# --- From RED ---
p_r_g = np.mean(next_green[is_red])
p_r_r = np.mean(next_red[is_red])
p_r_d = np.mean(next_doji[is_red])

# --- From DOJI ---
if np.any(is_doji):
    p_d_g = np.mean(next_green[is_doji])
    p_d_r = np.mean(next_red[is_doji])
    p_d_d = np.mean(next_doji[is_doji])
else:
    p_d_g = p_d_r = p_d_d = 0.0

print("\n" + "="*55)
print("TEST 2: 3-STATE CANDLE MARKOV CHAIN (1-Bar Forward)")
print("="*55)
print(f"Current GREEN -> Next GREEN: {p_g_g*100:.2f}%")
print(f"Current GREEN -> Next RED:   {p_g_r*100:.2f}%")
print(f"Current GREEN -> Next DOJI:  {p_g_d*100:.2f}%")
print(f"                 Total Check: {(p_g_g + p_g_r + p_g_d)*100:.2f}%")
print("-" * 40)
print(f"Current RED   -> Next GREEN: {p_r_g*100:.2f}%")
print(f"Current RED   -> Next RED:   {p_r_r*100:.2f}%")
print(f"Current RED   -> Next DOJI:  {p_r_d*100:.2f}%")
print(f"                 Total Check: {(p_r_g + p_r_r + p_r_d)*100:.2f}%")
print("-" * 40)
print(f"Current DOJI  -> Next GREEN: {p_d_g*100:.2f}%")
print(f"Current DOJI  -> Next RED:   {p_d_r*100:.2f}%")
print(f"Current DOJI  -> Next DOJI:  {p_d_d*100:.2f}%")
print(f"                 Total Check: {(p_d_g + p_d_r + p_d_d)*100:.2f}%")
print("="*55)


SyntaxError: unterminated string literal (detected at line 88) (252095780.py, line 88)

In [5]:
# Sweep cutoff periods to find optimal edge
periods = np.arange(5, 101, 5)
results = []

for p in periods:
    us_p = ultimate_smoother(close, p)
    slope_p = np.diff(us_p, prepend=us_p[0])
    inc_p = slope_p[:-1] > 0
    out = next_close_above[:-1]

    p_inc = out[inc_p].mean() if inc_p.sum() > 0 else 0.5
    p_dec = out[~inc_p].mean() if (~inc_p).sum() > 0 else 0.5

    results.append({
        'period': p,
        'p_above_inc': p_inc,
        'p_above_dec': p_dec,
        'edge_inc': p_inc - 0.5,
        'edge_dec': 0.5 - p_dec,
        'total_edge': (p_inc - p_dec),
        'n_inc': inc_p.sum(),
        'n_dec': (~inc_p).sum(),
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
print(f'\nBest total edge at period = {df_results.loc[df_results["total_edge"].idxmax(), "period"]}')

 period  p_above_inc  p_above_dec  edge_inc  edge_dec  total_edge  n_inc  n_dec
      5     0.442350     0.434415 -0.057650  0.065585    0.007935  45681  46451
     10     0.441246     0.435509 -0.058754  0.064491    0.005737  45614  46518
     15     0.442317     0.434451 -0.057683  0.065549    0.007866  45655  46477
     20     0.442971     0.433833 -0.057029  0.066167    0.009138  45538  46594
     25     0.442409     0.434374 -0.057591  0.065626    0.008034  45580  46552
     30     0.443637     0.433155 -0.056363  0.066845    0.010482  45659  46473
     35     0.443736     0.433054 -0.056264  0.066946    0.010682  45669  46463
     40     0.443168     0.433603 -0.056832  0.066397    0.009565  45714  46418
     45     0.443470     0.433310 -0.056530  0.066690    0.010160  45701  46431
     50     0.443220     0.433570 -0.056780  0.066430    0.009650  45632  46500
     55     0.443693     0.433093 -0.056307  0.066907    0.010600  45687  46445
     60     0.442850     0.433931 -0.057

In [ ]:
# === STRATEGY: always long when US increasing, always short when decreasing ===

CUTOFF_PERIOD = 20  # <-- CHANGE THIS to adjust cutoff frequency

us_strat = ultimate_smoother(close, CUTOFF_PERIOD)
us_slope_strat = np.diff(us_strat, prepend=us_strat[0])

signal = np.zeros(n_bars)
for i in range(3, n_bars):
    if us_slope_strat[i] > 0:
        signal[i] = 1
    elif us_slope_strat[i] < 0:
        signal[i] = -1

n_flips = int(np.sum(np.abs(np.diff(signal)) > 0))
n_long = int(np.sum(signal == 1))
n_short = int(np.sum(signal == -1))
print(f'Cutoff Period: {CUTOFF_PERIOD}')
print(f'Position flips: {n_flips:,}')
print(f'Bars long: {n_long:,} | Bars short: {n_short:,} | Bars flat: {int(np.sum(signal == 0)):,}')

In [7]:
bt = Backtester(
    ohlc=ohlc,
    indicies=indices,
    signal=signal,
    timeframe=timeframe,
    tick_value=tick_value,
    contract_multiplier=contract_multiplier,
    initial_equity=initial_equity,
    slippage_ticks=0,
    commission=5.0,
)
bt.full_backtest()
print(bt)


        Backtest Results from 2026-02-01 23:00:00.003520-06:00 to 2026-03-02 07:30:20.001880-06:00 (28 days 08:30:19.998360)
        Equity:     $100,000.00 -> $36,065.00
        Buy & Hold: $100,000.00 -> $93,162.50
        Buy & Hold Return (%): -6.84%
        Strategy Return (%):   -63.93%

        Total Trades:     12312.00
        Win Rate:         49.76%
        95% CI:           48.88% - 50.65%
        Avg. Win:         $112.26
        Avg. Loss:        $-121.55
        Ratio Win/Loss:   0.92
        Expected Value:   $-5.19

        Largest Win:      $2,882.50
        Largest Loss:     $-1,267.50
        Max Consec. Wins: 16
        Max Consec. Loss: 14

        Max Drawdown from 2026-02-02 20:15:11.007200-06:00 to 2026-02-27 14:40:03.003760-06:00 (24 days 18:24:51.996560)
        Max Drawdown:     $-71,150.00
        Max Drawdown (%): -70.32%

        Slippage:   0 ticks
        Commission: $5.00
        Contract Multiplier: $50.00
        Transaction Costs:   $61,560.00

   

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(indices, bt.equity_curve, linewidth=0.8, color='blue')
ax1.axhline(initial_equity, color='gray', linestyle='--', alpha=0.5)
ax1.set_ylabel('Equity ($)')
ax1.set_title(f'Ultimate Smoother Strategy - ES 250 Tick | Period={CUTOFF_PERIOD} | Always In (Long/Short) | $5 commission')
ax1.grid(True, alpha=0.3)

ax2.fill_between(indices, bt.drawdown_curve, 0, color='red', alpha=0.4)
ax2.set_ylabel('Drawdown ($)')
ax2.set_xlabel('Time')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [8]:
from Engine.Report import generate_report
generate_report(bt, 'backtest_report.html', open_browser=True)

Report saved: C:\Users\ryany\OneDrive\Desktop\Futures-PC\MacLink\RugbeatunoBacktestingSuite\backtest_report.html


'C:\\Users\\ryany\\OneDrive\\Desktop\\Futures-PC\\MacLink\\RugbeatunoBacktestingSuite\\backtest_report.html'